# 🚀 Ultimate 3D Multi-Modal Tracklet Pipeline (v13)
This is the complete, end-to-end solution for object classification using the KITTI data format. It fuses **3D LiDAR tracklet geometry** with **2D Camera visual features** and includes a comprehensive visualization suite.

### Pipeline Overview:
1. **Preprocessing**: Converts raw XML tracklets into a standardized format.
2. **Calibration Engine**: Calculates the Projection Matrix to bridge 3D and 2D spaces.
3. **Vision Branch**: Uses a pretrained Faster R-CNN ResNet-50 FPN to extract visual context.
4. **Fusion**: Concatenates spatial and visual features into temporal sequences.
4b. **Temporal FPN**: Multi-scale 1-D FPN (P3/P4/P5) with top-down lateral connections after the Conv1D prefix.
5. **Temporal Model**: A Bidirectional LSTM (Bi-LSTM) classifies the FPN-enhanced 32-frame sequence.
6. **Evaluation & Visualization**: Metrics (mAP), Confusion Matrix, Performance Curves, and Inference Overlays.

### Expected Data Layout:
```
data/
  Video_9/
    Video.xml
    sequence_images/        ← all frames as 0000000000.png, 0000000001.png, …
    calib_cam_to_cam.txt
    calib_imu_to_velo.txt
    calib_velo_to_cam.txt
  Video_10/ …
  Video_11/ …
  Video_12/ …
  Video_13/ …
```

In [4]:
import os
print("CWD:", os.getcwd())
print("__file__ would be:", os.path.abspath(""))

# Show what's actually in the current directory
print("\nContents of CWD:")
for item in sorted(os.listdir(".")):
    print(" ", item)

# Check if data exists relative to CWD
print("\ndata/ exists:", os.path.isdir("data"))

# Also try going up one level
print("../data/ exists:", os.path.isdir("../data"))

CWD: c:\Users\kumbi\OneDrive\Documents\GitHub\3D-ResNet-Temporal-Stacking-LSTM\src
__file__ would be: c:\Users\kumbi\OneDrive\Documents\GitHub\3D-ResNet-Temporal-Stacking-LSTM\src

Contents of CWD:
  model.ipynb
  model_v2.ipynb
  model_v3.ipynb
  model_v4.ipynb

data/ exists: False
../data/ exists: True


In [5]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import (average_precision_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

# ── Global Configuration ──────────────────────────────────────────────────────
TARGET_FRAMES   = 32
COORD_FEATURES  = 9
VISUAL_FEATURES = 1024

# Each entry: (xml_path, cam_to_cam, imu_to_velo, velo_to_cam)
# XML filename is Video.xml for every sequence; images live in sequence_images/
# return video.xml path, calib_cam_to_cam.txt path, calib_imu_to_velo.txt path, images directory path 
ALL_VIDEO_SEQ = [
    ("../data/Video_9/Video.xml",
     "../data/Video_9/calib_cam_to_cam.txt",
     "../data/Video_9/calib_imu_to_velo.txt",
     "../data/Video_9/calib_velo_to_cam.txt","../data/Video_9/sequence_images/","Video_9"),

    ("../data/Video_10/Video.xml",
     "../data/Video_10/calib_cam_to_cam.txt",
     "../data/Video_10/calib_imu_to_velo.txt",
     "../data/Video_10/calib_velo_to_cam.txt", "../data/Video_10/sequence_images/","Video_10"),

    ("../data/Video_11/Video.xml",
     "../data/Video_11/calib_cam_to_cam.txt",
     "../data/Video_11/calib_imu_to_velo.txt",
     "../data/Video_11/calib_velo_to_cam.txt","../data/Video_11/sequence_images/","Video_11"),

    ("../data/Video_12/Video.xml",
     "../data/Video_12/calib_cam_to_cam.txt",
     "../data/Video_12/calib_imu_to_velo.txt",
     "../data/Video_12/calib_velo_to_cam.txt","../data/Video_12/sequence_images/","Video_12"),

    ("../data/Video_13/Video.xml",
     "../data/Video_13/calib_cam_to_cam.txt",
     "../data/Video_13/calib_imu_to_velo.txt",
     "../data/Video_13/calib_velo_to_cam.txt","../data/Video_13/sequence_images/","Video_13"),
]

NUM_CLASSES   = 5
CLASS_MAP     = {'Car': 0, 'Van': 1, 'Truck': 2, 'Pedestrian': 3, 'Cyclist': 4}
INV_CLASS_MAP = {v: k for k, v in CLASS_MAP.items()}

In [6]:
# ── Dataset splits ───────────────────────────────────────────────────────────
# Index  Video    Role
#   0    Video_9   test
#   1    Video_10  test
#   2    Video_11  train
#   3    Video_12  train
#   4    Video_13  eval  (held-out, never seen during training)

test_set  = ALL_VIDEO_SEQ[0:2]   # Videos 9 & 10  – two sequences for testing
train_set = ALL_VIDEO_SEQ[2:4]   # Videos 11 & 12 – two sequences for training
eval_set  = ALL_VIDEO_SEQ[4:5]   # Video  13      – kept as list so every split is iterable

print(f'Train sequences : {len(train_set)}')
print(f'Test  sequences : {len(test_set)}')
print(f'Eval  sequences : {len(eval_set)}')

Train sequences : 2
Test  sequences : 2
Eval  sequences : 1


## 0. Path Sanity Check
Run this cell before building datasets to confirm every sequence folder and image directory is found.

In [7]:
def check_data_paths(img_root='data'):
    """Verify all sequence paths exist and report image counts per video."""
    all_ok = True
    for xml_path, cam_to_cam, imu_to_velo, velo_to_cam, images_dir, name in ALL_VIDEO_SEQ:
        print(xml_path, cam_to_cam, imu_to_velo, velo_to_cam, images_dir, name)
        video_name = name
        img_dir    = images_dir

        xml_ok   = os.path.isfile(xml_path)
        c2c_ok   = os.path.isfile(cam_to_cam)
        i2v_ok   = os.path.isfile(imu_to_velo)
        v2c_ok   = os.path.isfile(velo_to_cam)
        img_ok   = os.path.isdir(img_dir)
        n_images = len([f for f in os.listdir(img_dir) if f.endswith('.png')]) if img_ok else 0

        status = '✓' if all([xml_ok, c2c_ok, i2v_ok, v2c_ok, img_ok]) else '✗'
        if status == '✗':
            all_ok = False
        print(f'{status}  {video_name}  |  xml={xml_ok}  c2c={c2c_ok}  '
              f'i2v={i2v_ok}  v2c={v2c_ok}  imgs={n_images} ({img_dir})')

    if all_ok:
        print('\nAll paths verified ✓')
    else:
        print('\n⚠ One or more paths are missing — fix before running make_datasets().')

check_data_paths()

../data/Video_9/Video.xml ../data/Video_9/calib_cam_to_cam.txt ../data/Video_9/calib_imu_to_velo.txt ../data/Video_9/calib_velo_to_cam.txt ../data/Video_9/sequence_images/ Video_9
✓  Video_9  |  xml=True  c2c=True  i2v=True  v2c=True  imgs=270 (../data/Video_9/sequence_images/)
../data/Video_10/Video.xml ../data/Video_10/calib_cam_to_cam.txt ../data/Video_10/calib_imu_to_velo.txt ../data/Video_10/calib_velo_to_cam.txt ../data/Video_10/sequence_images/ Video_10
✓  Video_10  |  xml=True  c2c=True  i2v=True  v2c=True  imgs=22 (../data/Video_10/sequence_images/)
../data/Video_11/Video.xml ../data/Video_11/calib_cam_to_cam.txt ../data/Video_11/calib_imu_to_velo.txt ../data/Video_11/calib_velo_to_cam.txt ../data/Video_11/sequence_images/ Video_11
✓  Video_11  |  xml=True  c2c=True  i2v=True  v2c=True  imgs=437 (../data/Video_11/sequence_images/)
../data/Video_12/Video.xml ../data/Video_12/calib_cam_to_cam.txt ../data/Video_12/calib_imu_to_velo.txt ../data/Video_12/calib_velo_to_cam.txt ../da

## 1. Calibration Engine
Projects 3D LiDAR space ($tx, ty, tz$) into 2D pixel space ($u, v$).

In [8]:
def _parse_calib_file(path):
    """Return a dict of {key: np.array} from a KITTI calibration text file."""
    calib = {}
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or ':' not in line:
                continue
            key, vals = line.split(':', 1)
            calib[key.strip()] = np.array([float(v) for v in vals.split()])
    return calib


def get_projection_matrix(cam_to_cam_path, velo_to_cam_path):
    """Build the full 3×4 projection matrix  P = P2 @ R_rect @ T_velo→cam.

    Parses calibration files by *key name* so the function is robust to
    files with different numbers of header/comment lines.
    """
    # ── velo→cam extrinsic ────────────────────────────────────────────────────
    vc = _parse_calib_file(velo_to_cam_path)
    R_vc = vc['R'].reshape(3, 3)
    T_vc = vc['T'].reshape(3, 1)
    tr_velo_to_cam = np.vstack([np.hstack([R_vc, T_vc]), [0, 0, 0, 1]])  # 4×4

    # ── cam→cam rectification + projection ───────────────────────────────────
    cc = _parse_calib_file(cam_to_cam_path)
    r_rect = np.eye(4)
    r_rect[:3, :3] = cc['R_rect_00'].reshape(3, 3)
    p_rect_02 = cc['P_rect_02'].reshape(3, 4)

    return p_rect_02 @ r_rect @ tr_velo_to_cam   # 3×4


def project_3d_to_2d(p_matrix, x, y, z):
    """Project a single 3-D LiDAR point to 2-D pixel coordinates."""
    pt_3d = np.array([x, y, z, 1.0])
    pt_2d = p_matrix @ pt_3d
    if pt_2d[2] == 0:
        return 0, 0
    return int(pt_2d[0] / pt_2d[2]), int(pt_2d[1] / pt_2d[2])

## 2. Vision Feature Extractor
Extracts visual descriptors using Faster R-CNN FPN backbone.

In [9]:
class VisionExtractor:
    def __init__(self):
        full_model = fasterrcnn_resnet50_fpn(pretrained=True)
        self.backbone = full_model.backbone
        self.backbone.eval()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.backbone.to(self.device)
        self.transform = T.Compose([
            T.Resize((224, 224)), T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def extract(self, pil_img):
        img_t = self.transform(pil_img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            feats = self.backbone(img_t)
            return torch.mean(feats['0'], dim=[2, 3]).cpu().numpy().flatten()

## 3. KITTI Dataset Loader

**Key fixes vs v12:**
- `ALL_VIDEO_SEQ` now points to `Video.xml` (not `Video9.xml` etc.)
- `_parse_xml` records each pose's original `frame_idx` so the correct image filename is always used, regardless of temporal crop / padding.
- `_build_sequence` resolves images from `sequence_images/` using the stored `frame_idx`, not the local loop counter `fi`.

In [10]:
import xml.etree.ElementTree as ET
from tensorflow.keras.utils import to_categorical


class KITTIDataset:
    """Loads and processes one or more KITTI tracklet sequences.

    Parameters
    ----------
    sequences  : list of (xml_path, cam_to_cam, imu_to_velo, velo_to_cam)
    extractor  : VisionExtractor instance (shared across splits to save VRAM)
    img_root   : root folder containing Video_N sub-directories
                 frames expected at <img_root>/<Video_N>/sequence_images/<frame_idx:010d>.png
    augment    : if True, apply temporal jitter (train split only)
    """

    def __init__(self, sequences, extractor, img_root='data', augment=False):
        self.sequences = sequences
        self.extractor = extractor
        self.img_root  = img_root
        self.augment   = augment
        self.X, self.y = self._build()

    # ── public helpers ────────────────────────────────────────────────────────
    def __len__(self):
        return len(self.y)

    def as_arrays(self):
        """Return (X, y_onehot) ready to pass to model.fit()."""
        return self.X, to_categorical(self.y, num_classes=NUM_CLASSES)

    # ── internals ─────────────────────────────────────────────────────────────
    def _build(self):
        all_X, all_y = [], []
        for xml_path, cam_to_cam, _, velo_to_cam in self.sequences:
            p_mat     = get_projection_matrix(cam_to_cam, velo_to_cam)
            video_dir = os.path.dirname(xml_path)   # e.g. data/Video_11
            tracklets = self._parse_xml(xml_path)
            for label, frames in tracklets:
                if label not in CLASS_MAP:
                    continue
                seq = self._build_sequence(frames, p_mat, video_dir)
                all_X.append(seq)
                all_y.append(CLASS_MAP[label])
        return np.array(all_X, dtype=np.float32), np.array(all_y, dtype=np.int32)

    def _parse_xml(self, xml_path):
        """Return list of (class_label, [frame_dict, ...]) per tracklet.

        Each frame_dict now includes 'frame_idx' — the pose's position in the
        original video sequence — so _build_sequence can look up the correct
        image file even after temporal cropping or padding.
        """
        tree = ET.parse(xml_path)
        root = tree.getroot()
        tracklets = []
        for obj in root.findall('.//item'):
            label = obj.findtext('objectType', default='').strip()
            frames = []
            for frame_idx, pose in enumerate(obj.findall('.//item')):
                tx = float(pose.findtext('tx', '0'))
                ty = float(pose.findtext('ty', '0'))
                tz = float(pose.findtext('tz', '0'))
                rx = float(pose.findtext('rx', '0'))
                ry = float(pose.findtext('ry', '0'))
                rz = float(pose.findtext('rz', '0'))
                # Bounding-box dimensions live on the parent <item>
                w  = float(obj.findtext('w', '0'))
                h  = float(obj.findtext('h', '0'))
                l  = float(obj.findtext('l', '0'))
                frames.append(dict(tx=tx, ty=ty, tz=tz,
                                   rx=rx, ry=ry, rz=rz,
                                   w=w,   h=h,   l=l,
                                   frame_idx=frame_idx))   # ← real video frame index
            if frames:
                tracklets.append((label, frames))
        return tracklets

    def _build_sequence(self, frames, p_mat, video_dir):
        """Return a (TARGET_FRAMES, COORD_FEATURES + VISUAL_FEATURES) array.

        Images are loaded from:
            <img_root>/<Video_N>/sequence_images/<frame_idx:010d>.png
        where frame_idx is the original video frame number stored in each
        frame dict by _parse_xml.
        """
        # ── pad / truncate to TARGET_FRAMES ───────────────────────────────────
        if len(frames) >= TARGET_FRAMES:
            if self.augment:
                # Random temporal crop during training for jitter augmentation
                start = np.random.randint(0, len(frames) - TARGET_FRAMES + 1)
            else:
                start = 0
            frames = frames[start: start + TARGET_FRAMES]
        else:
            # Replicate last frame to reach TARGET_FRAMES
            pad    = [frames[-1]] * (TARGET_FRAMES - len(frames))
            frames = frames + pad

        video_name = os.path.basename(video_dir)   # e.g. Video_11

        seq = []
        for f in frames:
            # ── 9-D geometric feature ─────────────────────────────────────────
            coord = np.array([f['tx'], f['ty'], f['tz'],
                               f['rx'], f['ry'], f['rz'],
                               f['w'],  f['h'],  f['l']], dtype=np.float32)

            # ── 1024-D visual feature ─────────────────────────────────────────
            # Use f['frame_idx'] (original video frame #) NOT the loop counter,
            # so the correct file is found even after temporal cropping/padding.
            img_path = os.path.join(self.img_root,
                                    video_name,
                                    'sequence_images',
                                    f"{f['frame_idx']:010d}.png")
            if os.path.exists(img_path):
                pil_img = Image.open(img_path).convert('RGB')
                vis = self.extractor.extract(pil_img).astype(np.float32)
            else:
                vis = np.zeros(VISUAL_FEATURES, dtype=np.float32)

            seq.append(np.concatenate([coord, vis]))

        return np.stack(seq)   # (TARGET_FRAMES, 1033)


# ── Convenience factory ───────────────────────────────────────────────────────
def make_datasets(extractor, img_root='data'):
    """Instantiate train / test / eval KITTIDataset objects.

    Returns
    -------
    train_ds, test_ds, eval_ds : KITTIDataset
    """
    print('Building TRAIN dataset …')
    train_ds = KITTIDataset(train_set, extractor, img_root, augment=True)

    print('Building TEST  dataset …')
    test_ds  = KITTIDataset(test_set,  extractor, img_root, augment=False)

    print('Building EVAL  dataset …')
    eval_ds  = KITTIDataset(eval_set,  extractor, img_root, augment=False)

    print(f'  train={len(train_ds)}  test={len(test_ds)}  eval={len(eval_ds)} tracklets')
    return train_ds, test_ds, eval_ds

## 4. Temporal Model Architecture

## 4a. Temporal FPN Block
A **1-D Feature Pyramid Network** (FPN) inspired by the Faster R-CNN FPN head.
After the Conv1D + BatchNorm prefix has smoothed local features, the FPN applies
three parallel Conv1D *pyramid levels* (P3 / P4 / P5) with increasing dilation rates
to capture multi-scale temporal context, then merges them with lateral add connections
and upsampling — exactly mirroring the top-down pathway of a 2-D FPN.
The merged representation is projected back to 128 channels before the Bi-LSTM.

In [11]:
class TemporalFPNBlock(layers.Layer):
    """1-D Feature Pyramid Network block.

    Mirrors the Faster R-CNN FPN top-down pathway in the temporal domain:
      - Bottom-up:  three Conv1D branches with dilation 1 / 2 / 4  (P3/P4/P5)
      - Top-down:   upsample + lateral-add merge (P5 → P4 → P3)
      - Output:     single projection back to `out_channels`

    Input shape : (batch, frames, channels)
    Output shape: (batch, frames, out_channels)
    """

    def __init__(self, out_channels: int = 128, **kwargs):
        super().__init__(**kwargs)
        self.out_channels = out_channels

        # ── Bottom-up pyramid (P3 / P4 / P5) ────────────────────────────────
        self.p3_conv = layers.Conv1D(out_channels, 3, padding='same',
                                     dilation_rate=1, activation='relu', name='p3_conv')
        self.p4_conv = layers.Conv1D(out_channels, 3, padding='same',
                                     dilation_rate=2, activation='relu', name='p4_conv')
        self.p5_conv = layers.Conv1D(out_channels, 3, padding='same',
                                     dilation_rate=4, activation='relu', name='p5_conv')

        # ── Lateral 1×1 projections (match channels before adding) ──────────
        self.lat_p4  = layers.Conv1D(out_channels, 1, padding='same', name='lat_p4')
        self.lat_p3  = layers.Conv1D(out_channels, 1, padding='same', name='lat_p3')

        # ── Top-down merge convolutions (smooth after add) ───────────────────
        self.merge_p4 = layers.Conv1D(out_channels, 3, padding='same',
                                      activation='relu', name='merge_p4')
        self.merge_p3 = layers.Conv1D(out_channels, 3, padding='same',
                                      activation='relu', name='merge_p3')

        # ── Output projection ────────────────────────────────────────────────
        self.out_proj = layers.Conv1D(out_channels, 1, padding='same', name='fpn_out')
        self.bn_out   = layers.BatchNormalization(name='fpn_bn')

    def call(self, x, training=False):
        # Bottom-up: three parallel multi-scale branches
        p3 = self.p3_conv(x)   # fine scale   (dilation 1)
        p4 = self.p4_conv(x)   # medium scale (dilation 2)
        p5 = self.p5_conv(x)   # coarse scale (dilation 4)

        # Top-down: merge P5 → P4 (same sequence length; no spatial upsample needed)
        p4_td = self.merge_p4(self.lat_p4(p4) + p5)

        # Top-down: merge P4_td → P3
        p3_td = self.merge_p3(self.lat_p3(p3) + p4_td)

        # Project merged features and normalise
        out = self.bn_out(self.out_proj(p3_td), training=training)
        return out

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'out_channels': self.out_channels})
        return cfg


def build_model():
    """Builds the full pipeline:
      Input → Conv1D prefix → BatchNorm
           → TemporalFPNBlock (P3/P4/P5 multi-scale top-down)
           → Bi-LSTM × 2 → Dense → Dropout → Softmax
    """
    inp = layers.Input(shape=(TARGET_FRAMES, COORD_FEATURES + VISUAL_FEATURES),
                       name='sequence_input')

    # ── Prefix: local smoothing ───────────────────────────────────────────────
    x = layers.Conv1D(128, kernel_size=3, padding='same',
                      activation='relu', name='prefix_conv')(inp)
    x = layers.BatchNormalization(name='prefix_bn')(x)

    # ── Temporal FPN (Faster R-CNN FPN-style, adapted to 1-D sequences) ──────
    x = TemporalFPNBlock(out_channels=128, name='temporal_fpn')(x)

    # ── Temporal reasoning ───────────────────────────────────────────────────
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True),
                             name='bilstm_1')(x)
    x = layers.Bidirectional(layers.LSTM(64), name='bilstm_2')(x)

    # ── Classifier head ──────────────────────────────────────────────────────
    x = layers.Dense(128, activation='relu', name='fc_head')(x)
    x = layers.Dropout(0.5, name='dropout')(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

    model = models.Model(inp, out, name='tracklet_fpn_bilstm')
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

## 5. Visualisation Helpers
Reusable functions for training curves, confusion matrices, and inference overlays.

In [12]:
def plot_training_metrics(history):
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    ax[0].plot(history.history['accuracy'],     label='Train')
    ax[0].plot(history.history['val_accuracy'], label='Val')
    ax[0].set_title('Model Accuracy')
    ax[0].set_xlabel('Epoch'); ax[0].legend()

    ax[1].plot(history.history['loss'],     label='Train')
    ax[1].plot(history.history['val_loss'], label='Val')
    ax[1].set_title('Model Loss')
    ax[1].set_xlabel('Epoch'); ax[1].legend()

    plt.tight_layout(); plt.show()


def plot_cm(y_true, y_pred, title='Confusion Matrix'):
    y_true_labels = np.argmax(y_true, axis=1)
    y_pred_labels = np.argmax(y_pred, axis=1)
    cm   = confusion_matrix(y_true_labels, y_pred_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=list(CLASS_MAP.keys()))
    disp.plot(cmap='Blues')
    plt.title(title)
    plt.tight_layout(); plt.show()


def visualize_inference(img_path, p_matrix, tx, ty, tz, pred_idx, true_idx):
    """Project a predicted tracklet centre onto the image and display it."""
    img = cv2.imread(img_path)
    if img is None:
        print(f'Image not found: {img_path}')
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    u, v = project_3d_to_2d(p_matrix, tx, ty, tz)

    cv2.circle(img, (u, v), 8, (255, 0, 0), -1)
    color      = (0, 255, 0) if pred_idx == true_idx else (255, 0, 0)
    label_text = f"Pred: {INV_CLASS_MAP[pred_idx]} | True: {INV_CLASS_MAP[true_idx]}"
    cv2.putText(img, label_text, (u - 60, v - 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    plt.figure(figsize=(10, 6)); plt.imshow(img); plt.axis('off'); plt.show()

## 6. Training & Evaluation
Instantiate datasets from the assigned splits, train the model, then evaluate
on the held-out **test** split and the final **eval** split separately.

In [13]:
# ── 1. Initialise shared vision extractor ────────────────────────────────────
extractor = VisionExtractor()

# ── 2. Build datasets from the assigned sequence splits ──────────────────────
train_ds, test_ds, eval_ds = make_datasets(extractor, img_root='data')

X_train, y_train = train_ds.as_arrays()   # Videos 11 & 12
X_test,  y_test  = test_ds.as_arrays()    # Videos  9 & 10
X_eval,  y_eval  = eval_ds.as_arrays()    # Video   13

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}   y_test : {y_test.shape}')
print(f'X_eval : {X_eval.shape}   y_eval : {y_eval.shape}')

# ── 3. Build model ────────────────────────────────────────────────────────────
model = build_model()
model.summary()

# ── 4. Callbacks ─────────────────────────────────────────────────────────────
cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=5, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint('best_model.keras', monitor='val_loss',
                               save_best_only=True, verbose=1),
]

# ── 5. Train (validate on test split) ────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=cb,
)

c:\Users\kumbi\OneDrive\Documents\GitHub\3D-ResNet-Temporal-Stacking-LSTM\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\kumbi\OneDrive\Documents\GitHub\3D-ResNet-Temporal-Stacking-LSTM\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Building TRAIN dataset …


ValueError: too many values to unpack (expected 4)

## 7. Test Split Evaluation (Videos 9 & 10)

In [ ]:
plot_training_metrics(history)

print('\n── Test split (Videos 9 & 10) ──')
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'  Loss: {test_loss:.4f}  Accuracy: {test_acc:.4f}')

y_test_pred = model.predict(X_test)
plot_cm(y_test, y_test_pred, title='Confusion Matrix — Test Split')
print(classification_report(
    np.argmax(y_test,      axis=1),
    np.argmax(y_test_pred, axis=1),
    target_names=list(CLASS_MAP.keys())))

## 8. Final Evaluation on Held-Out Eval Split (Video 13)

In [ ]:
print('\n── Eval split (Video 13) ──')
eval_loss, eval_acc = model.evaluate(X_eval, y_eval, verbose=0)
print(f'  Loss: {eval_loss:.4f}  Accuracy: {eval_acc:.4f}')

y_eval_pred = model.predict(X_eval)
plot_cm(y_eval, y_eval_pred, title='Confusion Matrix — Eval Split')
print(classification_report(
    np.argmax(y_eval,      axis=1),
    np.argmax(y_eval_pred, axis=1),
    target_names=list(CLASS_MAP.keys())))

## 9. Visual Inference Overlays
Project predictions directly onto frames for qualitative verification.
Edit the index `SAMPLE_IDX` to inspect different tracklets.

In [ ]:
# ── Example: overlay prediction on the first test-split frame ───────────────
SAMPLE_IDX = 0   # change to inspect other tracklets

# Rebuild projection matrix for the first test sequence
xml_path, cam_to_cam, _, velo_to_cam = test_set[0]
p_matrix   = get_projection_matrix(cam_to_cam, velo_to_cam)
video_name = os.path.basename(os.path.dirname(xml_path))

# Retrieve the first frame's pose from the raw dataset
sample_frames = test_ds.sequences  # access raw sequence list if needed
pred_idx = int(np.argmax(y_test_pred[SAMPLE_IDX]))
true_idx = int(np.argmax(y_test[SAMPLE_IDX]))

# The frame_idx for the first frame of a tracklet is 0
frame_idx = 0
img_path  = os.path.join('data', video_name, 'sequence_images',
                         f'{frame_idx:010d}.png')

# Approximate centre coords from the feature array
tx, ty, tz = X_test[SAMPLE_IDX, 0, 0], X_test[SAMPLE_IDX, 0, 1], X_test[SAMPLE_IDX, 0, 2]

visualize_inference(img_path, p_matrix, tx, ty, tz, pred_idx, true_idx)